In [61]:
import torch
import torch.nn as nn
import torch.onnx as onnx
import os
import json
import shutil
from random import randint
from ml_runner_exporter.onnx_exporter import export_onnx

In [62]:
fixtures_path = "tests/fixtures"

if os.path.exists(fixtures_path):
    shutil.rmtree(fixtures_path)
os.makedirs(fixtures_path)

In [63]:
def get_output_size() -> int:
    return randint(2, 10) * 5

In [64]:
class SimpleLinearModel(nn.Module):
    def __init__(self, input_size, output_size, layer_num):
        super(SimpleLinearModel, self).__init__()
        self.layers = nn.ModuleList()
        if layer_num == 1:
            self.layers.append(nn.Linear(input_size, output_size))
        else:
            inter_output_size = get_output_size()
            self.layers.append(nn.Linear(input_size, inter_output_size))
            inter_input_size = inter_output_size
            for i in range(layer_num - 2):
                inter_output_size = get_output_size()
                self.layers.append(nn.Linear(inter_input_size, inter_output_size))
                inter_input_size = inter_output_size
            self.layers.append(nn.Linear(inter_input_size, output_size))

    def forward(self, x):
        # Pass input through the linear layer
        output = x
        for layer in self.layers:
            output = layer.forward(output)
        return output

In [65]:
# Dense layers with every activation type chained in between (relu, sigmoid,
# tanh, softmax). Linear/identity activation is intentionally excluded since
# it has no corresponding ONNX node to export from.
class ActivationModel(nn.Module):
    def __init__(self, input_size, output_size):
        super(ActivationModel, self).__init__()
        self.linear1 = nn.Linear(input_size, 8)
        self.act1_relu = nn.ReLU()
        self.linear2 = nn.Linear(8, 12)
        self.act2_sigmoid = nn.Sigmoid()
        self.linear3 = nn.Linear(12, 10)
        self.act3_tanh = nn.Tanh()
        self.linear4 = nn.Linear(10, output_size)
        self.act4_softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.linear1(x)
        x = self.act1_relu(x)
        x = self.linear2(x)
        x = self.act2_sigmoid(x)
        x = self.linear3(x)
        x = self.act3_tanh(x)
        x = self.linear4(x)
        x = self.act4_softmax(x)
        return x

In [66]:
# Two conv layers stacked directly, no activation or flatten in between -
# isolates conv-to-conv chaining (D3 -> D3 -> D3) on its own.
class SimpleConvOnlyModel(nn.Module):
    def __init__(self):
        super(SimpleConvOnlyModel, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(in_channels=4, out_channels=2, kernel_size=3, stride=1, padding=1)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        return x

In [67]:
# Conv -> ReLU (D3 activation) -> Flatten, with no dense layer after -
# isolates the D3 -> Flat transition on its own.
class ConvFlattenModel(nn.Module):
    def __init__(self):
        super(ConvFlattenModel, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3, stride=1, padding=1)
        self.act_relu = nn.ReLU()
        self.flatten = nn.Flatten()

    def forward(self, x):
        x = self.conv1(x)
        x = self.act_relu(x)
        x = self.flatten(x)
        return x

In [68]:
# The full pipeline: Conv2d -> ReLU -> Conv2d -> ReLU -> Flatten -> Linear ->
# Sigmoid -> Linear -> Tanh -> Linear -> Softmax. Exercises every layer type
# and activation together in one model.
class FullConvModel(nn.Module):
    def __init__(self, output_size):
        super(FullConvModel, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3, stride=1, padding=1)
        self.act1_relu = nn.ReLU()
        self.conv2 = nn.Conv2d(in_channels=4, out_channels=8, kernel_size=3, stride=1, padding=1)
        self.act2_relu = nn.ReLU()
        self.flatten = nn.Flatten()
        self.linear1 = nn.Linear(8 * 4 * 4, 20)
        self.act3_sigmoid = nn.Sigmoid()
        self.linear2 = nn.Linear(20, 15)
        self.act4_tanh = nn.Tanh()
        self.linear3 = nn.Linear(15, output_size)
        self.act5_softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.conv1(x)
        x = self.act1_relu(x)
        x = self.conv2(x)
        x = self.act2_relu(x)
        x = self.flatten(x)
        x = self.linear1(x)
        x = self.act3_sigmoid(x)
        x = self.linear2(x)
        x = self.act4_tanh(x)
        x = self.linear3(x)
        x = self.act5_softmax(x)
        return x

In [69]:
# A single RNN layer on its own, keeping only the final hidden state -
# isolates the base recurrent op (return_sequences=False, matching the
# Rust default). batch_first is left False (the default) so the input/output
# layout matches the ONNX RNN op's own (seq_len, batch, features) convention
# directly, with no Transpose nodes inserted around it.
#
# forward() returns h_n untouched (shape (1, 1, hidden_size)) rather than
# squeezing it down to (hidden_size,) - any reshape/squeeze here would show
# up as an extra ONNX node the exporter doesn't expect next to an RNN node.
# The trailing singleton dims disappear anyway once test_output is flattened.
class SimpleRNNOnlyModel(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(SimpleRNNOnlyModel, self).__init__()
        self.rnn = nn.RNN(input_size=input_size, hidden_size=hidden_size, batch_first=False)

    def forward(self, x):
        _, h_n = self.rnn(x)
        return h_n


# Same RNN layer, but returning every timestep's hidden state instead of
# just the final one - exercises return_sequences=True (the D2 output path)
# on both the exporter and the Rust runtime.
class SimpleRNNReturnSequencesModel(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(SimpleRNNReturnSequencesModel, self).__init__()
        self.rnn = nn.RNN(input_size=input_size, hidden_size=hidden_size, batch_first=False)

    def forward(self, x):
        output, _ = self.rnn(x)
        return output

In [70]:
# A single GRU layer on its own, final hidden state only - same shape as
# SimpleRNNOnlyModel but exercises the gate-splitting logic in
# GRULayerParser.gru_layer_from_onnx instead of the plain RNN path.
class SimpleGRUOnlyModel(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(SimpleGRUOnlyModel, self).__init__()
        self.gru = nn.GRU(input_size=input_size, hidden_size=hidden_size, batch_first=False)

    def forward(self, x):
        _, h_n = self.gru(x)
        return h_n


# Same GRU layer, returning every timestep's hidden state instead of just
# the final one.
class SimpleGRUReturnSequencesModel(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(SimpleGRUReturnSequencesModel, self).__init__()
        self.gru = nn.GRU(input_size=input_size, hidden_size=hidden_size, batch_first=False)

    def forward(self, x):
        output, _ = self.gru(x)
        return output

In [71]:
def export_model(name: str, model: nn.Module, input_shape):
    # Avoids the "exporting a model while it is in training mode" warning;
    # doesn't change behavior here since none of these models use
    # dropout/batchnorm, but it's the right default for exported fixtures.
    model.eval()

    tmp_model_path = "temporary_model.onnx"

    # input_shape is one of:
    #   int              -> flat feature count (dense-only models); batch
    #                        prepended as dim 0: (1, features)
    #   (C, H, W) tuple   -> conv models; batch prepended as dim 0:
    #                        (1, C, H, W)
    #   (seq_len, F) tuple -> RNN/GRU models; batch goes in the *middle*,
    #                        matching nn.RNN/nn.GRU's batch_first=False (and
    #                        the ONNX RNN/GRU op's own) layout: (seq_len, 1, F)
    if isinstance(input_shape, tuple) and len(input_shape) == 3:
        dummy_input_data = torch.randn(1, *input_shape)
    elif isinstance(input_shape, tuple) and len(input_shape) == 2:
        seq_len, feature_size = input_shape
        dummy_input_data = torch.randn(1, seq_len, feature_size)
    else:
        dummy_input_data = torch.randn(1, input_shape)

    onnx.export(model, dummy_input_data, tmp_model_path, export_params=True, opset_version=17, dynamo=False)

    with torch.no_grad():
        output = model(dummy_input_data)

    model_output = {
        "model": export_onnx(tmp_model_path),
        # Tensor::data on the Rust side is always flat (row-major) regardless
        # of TensorShape, so flatten both input and output fully here rather
        # than relying on tolist()[0], which would leave singleton/batch dims
        # in for conv and RNN/GRU shapes.
        "test_input": dummy_input_data.flatten().tolist(),
        "test_output": output.flatten().tolist(),
    }

    with open(os.path.join(fixtures_path, name), "w") as f:
        print(f"Exporting model: {name}")
        json.dump(model_output, f, indent=2)

In [72]:
fixtures = [
    {
        "name": "dense_simple_model.json",
        "model": SimpleLinearModel(10, 5, 1),
        "input_shape": 10,
    },
    {
        "name": "dense_long_model.json",
        "model": SimpleLinearModel(10, 5, 20),
        "input_shape": 10,
    },
    {
        "name": "dense_large_model.json",
        "model": SimpleLinearModel(100, 100, 5),
        "input_shape": 100,
    },
    {
        "name": "activation_all_types_model.json",
        "model": ActivationModel(10, 5),
        "input_shape": 10,
    },
    {
        "name": "conv_simple_model.json",
        "model": SimpleConvOnlyModel(),
        "input_shape": (1, 4, 4),
    },
    {
        "name": "conv_flatten_model.json",
        "model": ConvFlattenModel(),
        "input_shape": (1, 4, 4),
    },
    {
        "name": "conv_flatten_dense_activation_model.json",
        "model": FullConvModel(5),
        "input_shape": (1, 4, 4),
    },
    {
        "name": "rnn_simple_model.json",
        "model": SimpleRNNOnlyModel(input_size=3, hidden_size=5),
        "input_shape": (4, 3),  # (seq_len, input_size)
    },
    {
        "name": "rnn_return_sequences_model.json",
        "model": SimpleRNNReturnSequencesModel(input_size=3, hidden_size=5),
        "input_shape": (4, 3),
    },
    {
        "name": "gru_simple_model.json",
        "model": SimpleGRUOnlyModel(input_size=3, hidden_size=5),
        "input_shape": (4, 3),
    },
    {
        "name": "gru_return_sequences_model.json",
        "model": SimpleGRUReturnSequencesModel(input_size=3, hidden_size=5),
        "input_shape": (4, 3),
    },
]

In [73]:
for fixture in fixtures:
    export_model(fixture["name"], fixture["model"], fixture["input_shape"])

/tmp/ipykernel_12730/1795897682.py:25: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  onnx.export(model, dummy_input_data, tmp_model_path, export_params=True, opset_version=17, dynamo=False)
/home/david/miniconda3/envs/pytorch/lib/python3.14/site-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset9.py:4570: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with RNN_TANH can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  return _generic_rnn(


Exporting model: dense_simple_model.json
Exporting model: dense_long_model.json
Exporting model: dense_large_model.json
Exporting model: activation_all_types_model.json
Exporting model: conv_simple_model.json
Exporting model: conv_flatten_model.json
Exporting model: conv_flatten_dense_activation_model.json


ValueError: Unsupported layer type: Constant